# Credit Risk Decision Engine — Deployable Calibration and Policy

This notebook invokes the authoritative training pipeline: contract selection, development-only candidate comparison, fold-fitted calibration, policy-holdout threshold optimization, and the sole final-test evaluation.


In [1]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import pandas as pd

from src.pipeline.prediction_pipeline import PredictionPipeline
from src.pipeline.training_pipeline import run_training


## Full deployable training and untouched final-test evaluation


In [2]:
result = run_training()
metrics = result["metrics"]
metadata = result["metadata"]
display(pd.Series({
    "selected_model": metadata["model_name"],
    "selected_calibration": metadata["calibration"]["selected_method"],
    "approve_threshold": metadata["policy"]["approve_threshold"],
    "reject_threshold": metadata["policy"]["reject_threshold"],
    "final_test_roc_auc": metrics["calibrated"]["roc_auc"],
    "final_test_pr_auc": metrics["calibrated"]["pr_auc"],
    "final_test_brier": metrics["calibrated"]["brier_score"],
}))
display(pd.DataFrame(metadata["model_selection"]["candidates"]).T)
display(pd.DataFrame(metrics["policy_test_summary"]).T)


2026-08-10 21:04:54 | INFO | credit_risk.training | Loading training data from D:\Credit-Risk-Decision-Engine\data\raw\application_train.csv


2026-08-10 21:05:01 | INFO | credit_risk.training | Data validation passed with 2 diagnostics


2026-08-10 21:05:01 | INFO | credit_risk.training | Split rows: model=196806 policy=49202 final_test=61503


2026-08-10 21:05:39 | INFO | credit_risk.training | Selected xgboost from deployable-feature OOF comparison


2026-08-10 21:05:40 | INFO | credit_risk.training | Preprocessing fitted on model partition only


...

2026-08-10 21:06:23 | INFO | credit_risk.training | Selected illustrative policy thresholds approve=0.110 reject=0.120


2026-08-10 21:06:23 | INFO | credit_risk.training | Saved model artifacts and final-test metrics under artifacts/


selected_model           xgboost
selected_calibration    isotonic
approve_threshold           0.11
reject_threshold            0.12
final_test_roc_auc      0.692048
final_test_pr_auc       0.168643
final_test_brier        0.071207
dtype: object

,available,model_name,parameters,training_seconds,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,brier_score,confusion_matrix,top_10_percent_default_capture
Logistic Regression,True,logistic_regression,{},4.838344,0.5,0.919245,0.0,0.0,0.0,0.640067,0.132286,0.072782,"[[180913, 5], [15888, 0]]",0.201158
XGBoost,True,xgboost,"{'n_estimators': 500, 'learning_rate': 0.05, '...",16.043558,0.5,0.919276,0.666667,0.000126,0.000252,0.684041,0.163565,0.071458,"[[180917, 1], [15886, 2]]",0.252895
LightGBM,True,lightgbm,"{'n_estimators': 500, 'learning_rate': 0.05, '...",15.846924,0.5,0.919205,0.259259,0.000441,0.00088,0.680292,0.160244,0.071649,"[[180898, 20], [15881, 7]]",0.244776


,applicants,rate,observed_default_rate,mean_predicted_probability,total_expected_loss
APPROVE,48455.0,0.787848,0.057043,0.058423,1.051854e+09
MANUAL_REVIEW,2025.0,0.032925,0.114568,0.114900,7.257959e+07
REJECT,11023.0,0.179227,0.178627,0.172615,5.332482e+08


## Benchmark versus deployable performance


In [3]:
benchmark = metrics["benchmark_reference"]["final_test_metrics"]
deployable = metrics["calibrated"]
comparison = pd.DataFrame({
    "Research full-feature benchmark": {key: benchmark[key] for key in ["roc_auc", "pr_auc", "brier_score", "top_10_percent_default_capture"]},
    "Deployable application model": {key: deployable[key] for key in ["roc_auc", "pr_auc", "brier_score", "top_10_percent_default_capture"]},
})
display(comparison)


,Research full-feature benchmark,Deployable application model
roc_auc,0.768668,0.692048
pr_auc,0.260485,0.168643
brier_score,0.067006,0.071207
top_10_percent_default_capture,0.352266,0.260826


The deployable model intentionally sacrifices performance associated with unavailable features so that every production feature can be reproduced for a new applicant. Calibration and policy results must be reported honestly even when calibration does not improve the final-test Brier score.


## Saved-artifact inference for a completely new applicant


In [4]:
applicant = {
    "age": 34,
    "years_employed": 5,
    "family_members": 2,
    "number_of_children": 0,
    "annual_income": 202500,
    "requested_loan_amount": 406597.5,
    "loan_annuity": 24700.5,
    "goods_purchase_price": 351000,
    "credit_product_type": "Cash loans",
    "income_type": "Working",
    "housing_situation": "House / apartment",
    "owns_car": "No",
    "owns_property": "Yes",
}
prediction = PredictionPipeline().predict(applicant, include_explanations=True)
assert prediction.loc[0, "application_id"].startswith("APP-")
assert "source_record_id" not in prediction
display(prediction)


,application_id,probability,probability_percent,risk_band,recommendation,expected_loss,missing_input_feature_count,top_risk_reasons
0,APP-F2CCFFAB5F,0.140006,14.00061,HIGH,REJECT,34155.676969,0,"[{'feature': 'Loan-to-Repayment Ratio', 'contr..."


## Limitations and handoff

Expected loss uses illustrative LGD and requested credit as EAD. Thresholds are portfolio demonstrations, not customer-treatment rules. The sample is historical, no protected-class fairness audit or temporal validation is claimed, and reason codes describe association rather than causation.
